In [ ]:
import sys
import pandas as pd
from typing import List, Dict
from SPARQLWrapper import SPARQLWrapper, JSON

class WikiDataQueryResults:
    """
    A class that can be used to query data from Wikidata using SPARQL and return the results as a Pandas DataFrame or a list
    of values for a specific key.
    """
    def __init__(self, query: str):
        """
        Initializes the WikiDataQueryResults object with a SPARQL query string.
        :param query: A SPARQL query string.
        """
        self.user_agent = "WDQS-example Python/%s.%s" % (sys.version_info[0], sys.version_info[1])
        self.endpoint_url = "https://query.wikidata.org/sparql"
        self.sparql = SPARQLWrapper(self.endpoint_url, agent=self.user_agent)
        self.sparql.setQuery(query)
        self.sparql.setReturnFormat(JSON)

    def __transform2dicts(self, results: List[Dict]) -> List[Dict]:
        """
        Helper function to transform SPARQL query results into a list of dictionaries.
        :param results: A list of query results returned by SPARQLWrapper.
        :return: A list of dictionaries, where each dictionary represents a result row and has keys corresponding to the
        variables in the SPARQL SELECT clause.
        """
        new_results = []
        for result in results:
            new_result = {}
            for key in result:
                new_result[key] = result[key]['value']
            new_results.append(new_result)
        return new_results

    def _load(self) -> List[Dict]:
        """
        Helper function that loads the data from Wikidata using the SPARQLWrapper library, and transforms the results into
        a list of dictionaries.
        :return: A list of dictionaries, where each dictionary represents a result row and has keys corresponding to the
        variables in the SPARQL SELECT clause.
        """
        results = self.sparql.queryAndConvert()['results']['bindings']
        results = self.__transform2dicts(results)
        return results

    def load_as_dataframe(self) -> pd.DataFrame:
        """
        Executes the SPARQL query and returns the results as a Pandas DataFrame.
        :return: A Pandas DataFrame representing the query results.
        """
        results = self._load()
        return pd.DataFrame.from_dict(results)

In [ ]:
query = """
SELECT ?city ?cityLabel
WHERE {
  ?city wdt:P31 wd:Q515.
  ?city wdt:P17 ?country.
  ?country wdt:P30 wd:Q46.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 50
"""

In [ ]:
data_extracter = WikiDataQueryResults(query)
df = data_extracter.load_as_dataframe()
print(f"Sample of {len(df)} European cities:")
print(df.head(10))

# Add cinema query section
print("\n" + "="*60)
print("CINEMA SEARCH IN EUROPEAN CITIES")
print("="*60 + "\n")

# Simple query: just get individual cinemas in Europe (no aggregation)
cinemas_query = """
SELECT ?cinemaLabel ?countryLabel
WHERE {
  ?cinema wdt:P31 wd:Q41253.
  ?cinema wdt:P131+ ?country.
  ?country wdt:P31 wd:Q6256.
  ?country wdt:P30 wd:Q46.
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
LIMIT 500
"""

try:
    cinema_data = WikiDataQueryResults(cinemas_query)
    df_cinemas = cinema_data.load_as_dataframe()
    print(f"Found {len(df_cinemas)} cinemas in European countries\n")
    
    # Count cinemas by country manually
    country_counts = df_cinemas['countryLabel'].value_counts().head(20)
    print("Top countries by cinema count:")
    print(country_counts)
    
except Exception as e:
    error_str = str(e)[:300]
    print(f"Error: {error_str}")
    print("\nWikidata cinema data might be incomplete. Falling back to OSM data...")
